# MPI COMMUNICATION:

## 1. INTRODUCTION:
MPI is an acronym for message passing protocol, is a standard for communication between multiple processes in parallel programming. 

In MPI, each process has its own private memory, therefore processes cannot access directly each other’s memory, though they exchange data through messages.

MPI is required in High performance computing, where complex program is distributed among multiple nodes, in order to reduce computational time.

There are several MPI versions available such MPI1, MPI2, MPI3, MPI4, in this study MPI4 will be discussed.

## 2. MPI4PY:
MPI4PY is a MPI binding for python programming language, it allows python program to exploit multiple processors.

In order to get start with mpi4py, first step is to install the library.

python3 -m pip install mpi4py

Then in a new python script, the library should be imported at the top of the script, before using mpi4.

from mpi4py import MPI

To start open parallel session following command is used;

MPI.Init()

At the end, after all the computations and exchange of data is finished, following command is used to close the parallel session.

MPI.Finalize()

How ever in mpi4py when library is imported both opening and closing of parallel session are done automatically.

**Communicator:** In parallel programming, communicator holds the group of processes, by default all the processes are in MPI.COMM. WORLD.

**Rank**, is the unique id associated to each process in a group, starting from zero. Whereas **size** refers to the total number of processes inside a group.

To get the rank of process id following command is used;  
comm.Get_rank()

To get the size of group, following command is used;  
comm.Get_size()

For sending and receiving messages between processes two types of communication used, blocking communication and non-blocking communication, either it’s a blocking communication or a non-blocking communication two types of communication functions are used;

**Lower Case:**  
**comm.send()** - used to send data in blocking communication.    
**comm.recv()** – used to receive data in blocking communication.  
**comm.isend()** – used to send data in non-blocking communication.  
**comm.irecv()** – used to receive data in non-blocking communication.  

In lower case communication function, data is sent as python object, mpi4py automatically convert object to bytes, this is known is pickling. On the receiving side the object is reconstructed, this is known as unpickling.

For every message this process is followed, this creates an overhead, and for large arrays, the process becomes slow.

**Upper Case:**    
**comm.Send()** - used to send data in blocking communication.  
**comm.Recv()** – used to receive data in blocking communication.  
**comm.Isend()** – used to send data in non-blocking communication.  
**comm.Irecv()** – used to receive data in non-blocking communication.  

The upper-case communication function works directly on memory buffer; therefore, it is much faster than the lower case.

## 3. Point to Point Communication:
It is the basic communication, that is performed between to processes that lie with in the same communicator. Process A sends the data and process B receives the data and vice versa. For point-to-point communication, both the source process and destination process must lie within same communicator. Within communicator source and destination processes are identified by their ranks.

There are two types of communication within the umbrella of point-to-point communication, that are;    
a.	Blocking communication.  
b.	Non-Blocking communication.  

**Blocking Communication:**  
In blocking communication, sender has to wait until the data is sent before reusing the buffer or performing other calculations. Similarly, receiver has also to wait until the data is received, this can cause allocated computational resources to go waste.

The communication can be further divided into two modes;
a.	Synchronous  
b.	Standard or A synchronous  

In **synchronous communication**, the message is transferred directly from senders’ application buffer to receivers’ application buffer, the send operations are completed only when the matching receives has been initiated.

On the contrary, in **standard or Asynchronous communication mode**, message is copied into system internal buffer, send operation is completed when the message is safely copied into system internal buffer even if the receive operation is still not started. Then it’s the system buffer, whose responsibility is to transfer the data to receiver memory, when matching receive is posted.

For blocking, standard point to point communication, command syntax is reported below;

**Blocking send:**  
comm.Send(buf, dest, tag)  
buf- NumPy array or buffer containing data to send.  
dest- rank of destination process.  
tag- message identifier.  

**Blocking Receive:**  
comm.recv(buf,source,tag,status)  
buf- buffer to store incoming data.  
source- rank of source process.  
tag- message identifier.  
status – incoming message information.  
To read status information, following commands can be used;  
status.Get_source()  
status.Get_tag()  
status.Get_error()  
In mpi4py there is no error code argument, it means it used python exception for error handling.

An example of send and receive data between two processes using blocking communication is reported below;


In [ ]:
# P2P Blocking Communication
import numpy as np
from mpi4py import MPI


comm = MPI.COMM_WORLD
my_id = comm.Get_rank()
nproc = comm.Get_size()

if my_id == 0:
    status = MPI.Status()
    A = np.array(2.0, dtype='d')
    B = np.empty(1, dtype='d')
    comm.Send(A,1,10)
    comm.Recv(B,1,11,status)
    print(f"My rank is {my_id} & I sent data  to process 1 sucessfully")
    print(f"my rank is {my_id}: and i received Data = {B}")
    print(f"Message source = {status.Get_source()}")
    print(f"Message tag    = {status.Get_tag()}")

elif my_id == 1:
    status = MPI.Status()
    B = np.empty(1, dtype='d')
    A = np.array(3.0, dtype='d')
    comm.Recv(B,0, 10,status=status)
    comm.Send(A,0,11)
    print(f"My rank is {my_id} & I sent data  to process 0 sucessfully")
    print(f"my rank is {my_id}: and i received Data = {B}")
    print(f"Message source = {status.Get_source()}")
    print(f"Message tag    = {status.Get_tag()}")


zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -n 2 python P2P.py  
My rank is 0 & I sent data  to process 1 sucessfully  
My rank is 1 & I sent data  to process 0 sucessfully  
my rank is 1: and i received Data = [2.]  
Message source = 0  
Message tag    = 10  
my rank is 0: and i received Data = [3.]  
Message source = 1  
Message tag    = 11  

In [ ]:
#P2P Non Blocking Communication.
import numpy as np
from mpi4py import MPI


comm = MPI.COMM_WORLD
my_id = comm.Get_rank()
nproc = comm.Get_size()

if my_id == 0:
    status = MPI.Status()
    A = np.array(2.0, dtype='d')
    B = np.empty(1, dtype='d')
    req1 = comm.Isend(A,1,10)
    req2 = comm.Irecv(B,1,11)
    MPI.Request.Waitall([req1, req2])
    print(f"My rank is {my_id} & I sent data  to process 1 sucessfully")
    print(f"my rank is {my_id}: and i received Data = {B}")

elif my_id == 1:
    status = MPI.Status()
    B = np.empty(1, dtype='d')
    A = np.array(3.0, dtype='d')
    req1 = comm.Isend(A,0,11)
    req2 = comm.Irecv(B,0, 10)
    MPI.Request.Waitall([req1, req2])
    print(f"My rank is {my_id} & I sent data  to process 0 sucessfully")
    print(f"my rank is {my_id}: and i received Data = {B}")



zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -n 2 python P2PNonBlock.py  
My rank is 0 & I sent data  to process 1 sucessfully  
My rank is 1 & I sent data  to process 0 sucessfully  
my rank is 0: and i received Data = [3.]  
my rank is 1: and i received Data = [2.]  

## 4. Comparison between Fortran and python - P2P:
| Operation             | Fortran MPI        | Python mpi4py              |
|----------------------|--------------------|----------------------------|
| Blocking send        | `MPI_SEND`         | `comm.send()`              |
| Blocking receive     | `MPI_RECV`         | `comm.recv()`              |
| Non-blocking send    | `MPI_ISEND`        | `comm.Isend()`             |
| Non-blocking receive | `MPI_IRECV`        | `comm.Irecv()`             |
| Wait single          | `MPI_WAIT`         | `req.wait()`               |
| Wait multiple        | `MPI_WAITALL`      | `MPI.Request.Waitall()`    |
| Status               | Output argument    | `MPI.Status()` object      |
| Request handling     | Passed by reference| Returned object            |